In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "maharashtra_agriculture_dataset.xlsx"

if not DATA_PATH.exists():
    search_roots = [PROJECT_ROOT, PROJECT_ROOT.parent]
    matches = [
        path
        for root in search_roots
        for path in root.rglob("maharashtra_agriculture_dataset.xlsx")
    ]
    if matches:
        DATA_PATH = matches[0]
    else:
        raise FileNotFoundError(
            f"Dataset not found under {search_roots}. "
            "Place the Excel file in data/raw or update DATA_PATH to its actual location."
        )

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(DATA_PATH)

df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

display(df.head())

,record_id,district,city,latitude,longitude,soil_type,soil_color,soil_ph,nitrogen_kg_ha,phosphorus_kg_ha,...,production_kg,market_price_rs_kg,total_cost_rs,revenue_rs,profit_rs,drought_risk_pct,flood_risk_pct,overall_risk_pct,recommended_crop,recommendation_score
0,1,Aurangabad,Chhatrapati Sambhajinagar,19.88,75.34,Black,Black,7.82,262.9,8.3,...,23856.2,64.8,817651.4,1546569.3,728918.0,16.5,21.1,26.8,Gram,88.9
1,2,Sangli,Sangli,16.85,74.58,Black,Black,6.95,248.6,24.6,...,64257.4,101.6,964374.1,6530727.3,5566353.1,17.8,24.6,28.4,Tur,86.5
2,3,Gondia,Gondia,21.46,80.20,Alluvial,Dark Brown,6.68,233.8,25.5,...,1246675.0,12.9,2393831.0,16051010.4,13657179.4,19.9,27.6,31.6,Rice,82.9
3,4,Gadchiroli,Gadchiroli,20.18,79.99,Red Loamy,Red,5.85,117.8,21.5,...,11167.6,65.0,181452.6,726359.5,544907.0,0.0,46.5,35.6,Rice,77.3
4,5,Parbhani,Parbhani,19.27,76.78,Black,Black,6.81,183.7,15.0,...,5016.5,65.3,125770.1,327441.6,201671.5,0.0,32.1,25.3,Groundnut,81.1


In [4]:
numeric_cols = df.select_dtypes(include=np.number).columns

summary = df[numeric_cols].describe().T

summary["variance"] = df[numeric_cols].var()
summary["skewness"] = df[numeric_cols].skew()
summary["kurtosis"] = df[numeric_cols].kurtosis()

display(summary)

summary.to_csv(
    REPORTS_DIR / "statistical_descriptive_summary.csv"
)

,count,mean,std,min,25%,50%,75%,max,variance,skewness,kurtosis
record_id,2500.0,1.250500e+03,7.218322e+02,1.00,6.257500e+02,1250.50,1.875250e+03,2500.00,5.210417e+05,0.000000,-1.200000
latitude,2500.0,1.944545e+01,1.421657e+00,16.17,1.852000e+01,19.72,2.070000e+01,21.46,2.021107e+00,-0.617271,-0.531422
longitude,2500.0,7.595234e+01,2.178440e+00,72.77,7.402000e+01,75.88,7.730000e+01,80.20,4.745600e+00,0.325653,-0.943757
soil_ph,2500.0,7.008216e+00,6.788195e-01,5.00,6.730000e+00,7.12,7.510000e+00,8.00,4.607959e-01,-0.887501,0.291753
nitrogen_kg_ha,2500.0,2.050718e+02,5.280059e+01,60.00,1.740750e+02,213.95,2.420000e+02,350.00,2.787902e+03,-0.530539,-0.117276
phosphorus_kg_ha,2500.0,2.038096e+01,7.427623e+00,5.00,1.547500e+01,20.40,2.550000e+01,46.20,5.516959e+01,0.040081,-0.265126
potassium_kg_ha,2500.0,2.726781e+02,8.044470e+01,60.00,2.237750e+02,282.50,3.311000e+02,472.60,6.471350e+03,-0.471784,-0.186994
organic_carbon_pct,2500.0,6.980040e-01,2.065304e-01,0.15,5.600000e-01,0.70,8.400000e-01,1.39,4.265480e-02,-0.099571,-0.239484
soil_moisture_pct,2500.0,4.005056e+01,1.166103e+01,15.00,3.200000e+01,39.70,4.760000e+01,85.70,1.359797e+02,0.227772,-0.063833
electrical_conductivity_ds_m,2500.0,7.417000e-01,3.422563e-01,0.10,5.000000e-01,0.73,9.700000e-01,2.18,1.171394e-01,0.255784,-0.160886


In [5]:
target_col = next(
    (c for c in df.columns if "yield" in c or "productivity" in c),
    None
)

if target_col is None:
    print("No yield column detected. Set target_col manually.")
else:
    values = pd.to_numeric(
        df[target_col], errors="coerce"
    ).dropna()

    if len(values) >= 2:
        mean = values.mean()
        sem = stats.sem(values)

        lower, upper = stats.t.interval(
            confidence=0.95,
            df=len(values) - 1,
            loc=mean,
            scale=sem
        )

        print("Target:", target_col)
        print("Sample size:", len(values))
        print("Mean:", round(mean, 3))
        print("Standard deviation:", round(values.std(), 3))
        print("95% confidence interval:", (lower, upper))

Target: yield_kg_acre
Sample size: 2500
Mean: 14083.402
Standard deviation: 21867.947
95% confidence interval: (np.float64(13225.779066251112), np.float64(14941.025333748888))


In [6]:
numeric_cols = df.select_dtypes(include=np.number).columns
results = []

for i, col1 in enumerate(numeric_cols):
    for col2 in numeric_cols[i + 1:]:
        pair = df[[col1, col2]].dropna()

        if (
            len(pair) >= 3
            and pair[col1].nunique() > 1
            and pair[col2].nunique() > 1
        ):
            r, p = stats.pearsonr(
                pair[col1], pair[col2]
            )

            results.append({
                "Feature 1": col1,
                "Feature 2": col2,
                "Pearson Correlation": r,
                "P-value": p,
                "Sample Size": len(pair)
            })

correlation_results = pd.DataFrame(results)

if not correlation_results.empty:
    correlation_results["Significant at 0.05"] = (
        correlation_results["P-value"] < 0.05
    )

    display(
        correlation_results.sort_values("P-value")
    )

    correlation_results.to_csv(
        REPORTS_DIR / "statistical_correlation_tests.csv",
        index=False
    )
else:
    print("No valid numeric feature pairs.")

,Feature 1,Feature 2,Pearson Correlation,P-value,Sample Size,Significant at 0.05
263,crop_duration_days,yield_kg_acre,0.703568,0.000000e+00,2500,True
306,yield_kg_acre,production_kg,0.845570,0.000000e+00,2500,True
231,rainfall_mm,humidity_pct,0.699112,0.000000e+00,2500,True
336,revenue_rs,profit_rs,0.998258,0.000000e+00,2500,True
330,total_cost_rs,revenue_rs,0.650495,9.182886e-301,2500,True
...,...,...,...,...,...,...
225,avg_temperature_c,revenue_rs,-0.000359,9.856775e-01,2500,False
237,rainfall_mm,production_kg,-0.000298,9.881087e-01,2500,False
108,nitrogen_kg_ha,fertilizer_kg_acre,-0.000103,9.958851e-01,2500,False
285,fertilizer_kg_acre,pesticide_l_acre,0.000070,9.971927e-01,2500,False


In [7]:
crop_col = next(
    (c for c in df.columns if "crop" in c),
    None
)

if target_col and crop_col:
    analysis_df = df[[crop_col, target_col]].copy()
    analysis_df[target_col] = pd.to_numeric(
        analysis_df[target_col], errors="coerce"
    )
    analysis_df = analysis_df.dropna()

    groups = [
        group[target_col].values
        for _, group in analysis_df.groupby(crop_col)
        if len(group) >= 2
    ]

    if len(groups) >= 2:
        statistic, p_value = stats.kruskal(*groups)

        print("Test: Kruskal-Wallis")
        print("Groups:", len(groups))
        print("Statistic:", statistic)
        print("P-value:", p_value)

        if p_value < 0.05:
            print("Evidence of differences between at least some groups.")
        else:
            print("Insufficient evidence of group differences.")
    else:
        print("Not enough crop groups for testing.")
else:
    print("Set crop_col and target_col using your dataset.")

Test: Kruskal-Wallis
Groups: 19
Statistic: 2427.5897817581954
P-value: 0.0
Evidence of differences between at least some groups.
